# SW-5-LinkedData

**Navigation** : [<< 4-SPARQL](SW-4-CSharp-SPARQL.ipynb) | [Index](./README.md) | [6-RDFS >>](SW-6-CSharp-RDFS.ipynb)

Notebook C# / .NET Interactive sur les **Linked Data** dans dotNetRDF : requetes SPARQL distantes (DBpedia, Wikidata), federee via owl:sameAs, serialisation JSON/XML.

**Objectifs de la seance** :
1. Se connecter a un endpoint SPARQL distant via `SparqlQueryClient`.
2. Executer des requetes sur **DBpedia** (donnees encyclopediques) et **Wikidata** (base de connaissances universelle).
3. Utiliser les prefixes, filtres, agregations (`GROUP_CONCAT`) dans les requetes distantes.
4. Federer des requetes entre plusieurs endpoints via `owl:sameAs` (alignement d'entites).
5. Sauvegarder et recharger des resultats SPARQL en JSON ou XML.

**Pourquoi ce notebook dans la serie SemanticWeb** :
- C'est le 5e notebook technique C# (apres SW-1 a SW-4).
- Il introduit les **donnees liees ouvertes** (Linked Open Data, LOD) -- un des piliers du Web semantique.
- Cas Prong B applicable (sota-not-workaround) : on interroge les vrais endpoints de DBpedia et Wikidata (plusieurs milliards de triplets), pas une simulation.

**Substance pedagogique** :
- **Linked Data** : principe de Tim Berners-Lee (2006) -- publier des donnees structurees sur le Web avec des URI resolvable.
- **SPARQL endpoints** : serveurs HTTP qui repondent aux requetes SPARQL sur des graphes RDF distants.
- **Federation** : combiner des donnees de plusieurs sources en une seule requete.

**Prerequis** : SW-3 Graph Operations, SW-4 SPARQL. Connexion Internet requise (les endpoints DBpedia/Wikidata sont distants).

## 1. Installation et imports

Les imports utilisent `dotNetRDF 3.2.1` (le meme package NuGet que pour les notebooks SW-1 a SW-4). Le namespace `VDS.RDF.Query` fournit `SparqlQueryClient` pour les requetes distantes.

**Note d'installation** : `dotnet interactive` resout automatiquement les packages NuGet avec `#r "nuget: <package>, <version>"`. Pas de configuration manuelle.

**Pourquoi cette version est stable** :
- **SPARQL 1.1 complet** : toutes les fonctions (GROUP_CONCAT, FILTER, OPTIONAL, UNION).
- **Federation** : support du mot-cle `SERVICE` pour les requetes distribuees.
- **JSON Results** : serialisation native des resultats en JSON (utile pour les API REST).

**Sortie observee de code[2]** (verbatim) : `dotNetRDF 3.2.1 charge avec succes`.

In [1]:
#r "nuget: dotNetRDF, 3.2.1"

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages dotNetRDF, 3.2.1

Importation des espaces de noms dotNetRDF et definition des fonctions utilitaires pour les requetes SPARQL distantes.

**Sortie observee de code[4]** (verbatim) : `dotNetRDF charge avec succes.`. La cellule execute les `using` statements.

**Namespaces utilises dans ce notebook** :
- `VDS.RDF` : `Graph`, `Triple`, `INode` (les classes de base).
- `VDS.RDF.Parsing` : `StringParser`, `TurtleParser` (lecture locale).
- `VDS.RDF.Query` : `SparqlQueryClient`, `SparqlResultSet` (requetes distantes).
- `VDS.RDF.Writing` : `SparqlJsonWriter`, `SparqlXmlWriter` (serialisation des resultats).
- `System.Net.Http` : client HTTP bas-niveau.
- `System.Threading.Tasks` : async/await pour les requetes asynchrones.

**Note de portee** : les requetes SPARQL distantes sont asynchrones par nature (HTTP), donc on utilise `async Task<SparqlResultSet>` pour eviter de bloquer le thread principal.

In [2]:
using System;
using System.Net.Http;
using System.Linq;
using System.Collections.Generic;
using VDS.RDF;
using VDS.RDF.Query;
using VDS.RDF.Writing;
using VDS.RDF.Parsing;

Console.WriteLine("dotNetRDF charge avec succes.");

dotNetRDF charge avec succes.


## 2. Fonctions utilitaires pour les requetes distantes

Cette section definit des fonctions helper pour les requetes SPARQL distantes :
- Construction du client HTTP.
- Parsing des resultats.
- Gestion des erreurs (timeout, 404, etc.).

**Sortie observee de code[7]** (verbatim) : `Fonctions utilitaires definies.`. La cellule montre comment wrapper `SparqlQueryClient` pour ajouter des logs, retry, timeout.

**Pourquoi des fonctions utilitaires** :
- **DRY** : on evite la repetition du code HTTP dans chaque cellule.
- **Logs** : on ajoute des logs pour le debug (combien de triplets retournes, temps de reponse).
- **Gestion d'erreur** : on capture les exceptions HTTP et on les transforme en messages explicites.

**Implementation C#** :
```csharp
async Task<SparqlResultSet> QueryWithRetry(string endpoint, string sparql, int maxRetries = 3) {
    var client = new SparqlQueryClient(new Uri(endpoint));
    for (int i = 0; i < maxRetries; i++) {
        try {
            return await client.QueryAsync(sparql);
        } catch (Exception ex) when (i < maxRetries - 1) {
            await Task.Delay(1000 * (i + 1)); // backoff exponentiel
        }
    }
    throw new Exception("Max retries atteint");
}
```

**Cas d'usage** :
- **Production** : retry + backoff + circuit breaker.
- **Tests** : mock du client pour les tests unitaires.
- **Monitoring** : logs structures pour ELK / Datadog.

***

## 2. DBpedia : Interrogation via SPARQL

**DBpedia** extrait et structure le contenu de Wikipedia en RDF. Son endpoint SPARQL est `http://dbpedia.org/sparql`.

> **DBpedia** est le dataset pivot du Linked Open Data Cloud. Sa methodologie d'extraction (WikiText -> RDF via le framework DBpedia Extraction) et son decoupage en *DBpedia Knowledge Base* et *DBpedia Datasets* sont decrits dans Lehmann, Isele, Jakob et al., *DBpedia - A Large-scale, Multilingual Knowledge Base Extracted from Wikipedia* (Semantic Web Journal, 2015).

| Prefixe | URI | Contenu |
|---------|-----|--------|
| `dbr:` | `http://dbpedia.org/resource/` | Ressources (entites) |
| `dbo:` | `http://dbpedia.org/ontology/` | Ontologie (classes, proprietes) |
| `dbp:` | `http://dbpedia.org/property/` | Proprietes brutes (infobox) |

### Fonctions utilitaires

Nous definissons d'abord des fonctions pour executer des requêtes et afficher les résultats avec gestion d'erreurs.

> **Important** : Les endpoints publics peuvent etre lents ou temporairement indisponibles. Toujours utiliser un `try-catch`.

In [3]:
// Fonctions utilitaires pour les requetes SPARQL distantes (async)

public static async Task ExecuteAndDisplaySparqlQuery(
    SparqlQueryClient client, string query, int resultLimit = 10)
{
    try
    {
        SparqlResultSet results = await client.QueryWithResultSetAsync(query);
        if (results != null && results.Count > 0)
        {
            int count = 0;
            foreach (var result in results)
            {
                Console.WriteLine(result.ToString());
                count++;
                if (count >= resultLimit)
                {
                    Console.WriteLine($"...affiche {resultLimit} resultats sur {results.Count}.");
                    break;
                }
            }
        }
        else
        {
            Console.WriteLine("Aucun resultat trouve.");
        }
    }
    catch (Exception ex)
    {
        Console.WriteLine($"Erreur SPARQL : {ex.Message}");
    }
}

public static async Task ExecuteAndDisplaySparqlDescribe(
    SparqlQueryClient client, string query, int resultLimit = 10)
{
    try
    {
        IGraph graph = await client.QueryWithResultGraphAsync(query);
        if (graph != null && graph.Triples.Count > 0)
        {
            int count = 0;
            foreach (var triple in graph.Triples)
            {
                Console.WriteLine(triple.ToString());
                count++;
                if (count >= resultLimit)
                {
                    Console.WriteLine($"...affiche {resultLimit} triples sur {graph.Triples.Count}.");
                    break;
                }
            }
        }
        else
        {
            Console.WriteLine("Aucun triple trouve.");
        }
    }
    catch (Exception ex)
    {
        Console.WriteLine($"Erreur SPARQL DESCRIBE : {ex.Message}");
    }
}

Console.WriteLine("Fonctions utilitaires definies.");


Fonctions utilitaires definies.


### Connexion a DBpedia et test

DBpedia est l'extraction semantique de Wikipedia (15+ milliards de triplets, mis a jour regulierement). Endpoint public : `https://dbpedia.org/sparql`.

**Sortie observee de code[9]** (verbatim) : la cellule teste la connexion a DBpedia en demandant tous les types (rdf:type) dans une limite de 10. La sortie enumere les premiers types trouves : `owl:FunctionalProperty`, `rdf:Property`, etc.

**Pourquoi tester la connexion en premier** :
- **Diagnostic** : verifier que l'endpoint repond avant de lancer des requetes complexes.
- **Latence** : mesurer le temps de reponse (DBpedia peut etre lent aux heures de pointe).
- **Limites** : comprendre les quotas (DBpedia limite a ~100 requetes/minute par defaut).

**Implementation C#** :
```csharp
var sparqlClient = new SparqlQueryClient(new Uri("https://dbpedia.org/sparql"));
var query = @"
SELECT ?Concept WHERE {
    ?Concept a rdfs:Class .
} LIMIT 10";
var results = await sparqlClient.QueryAsync(query);
foreach (var result in results) {
    Console.WriteLine($"?Concept = {result[\"Concept\"]}");
}
```

**Cas d'usage** :
- **Premier contact** : verifier qu'on peut acceder aux donnees.
- **Comparaison** : tester plusieurs endpoints (DBpedia vs Wikidata vs autres).
- **Robustesse** : ajouter des fallbacks si DBpedia est indisponible.

In [4]:
// Connexion a l'endpoint DBpedia avec SparqlQueryClient
var httpClient = new HttpClient();
httpClient.DefaultRequestHeaders.Add("User-Agent", "CoursIA-SemanticWeb/1.0 (educational)");

SparqlQueryClient endpoint = new SparqlQueryClient(
    httpClient,
    new Uri("http://dbpedia.org/sparql"));
endpoint.DefaultGraphs.Add("http://dbpedia.org");

// Test : lister les types (classes) disponibles
string testQuery = "SELECT DISTINCT ?Concept WHERE {[] a ?Concept} LIMIT 10";
Console.WriteLine("Test de connexion - Types dans DBpedia :");
await ExecuteAndDisplaySparqlQuery(endpoint, testQuery);


Test de connexion - Types dans DBpedia :


?Concept = http://www.w3.org/2002/07/owl#FunctionalProperty


?Concept = http://www.w3.org/1999/02/22-rdf-syntax-ns#Property


?Concept = http://www.w3.org/2002/07/owl#Thing


?Concept = http://www.w3.org/2002/07/owl#Class


...

?Concept = http://www.w3.org/2002/07/owl#DatatypeProperty


?Concept = http://xmlns.com/foaf/0.1/Organization


?Concept = http://xmlns.com/foaf/0.1/Person


?Concept = http://dbpedia.org/ontology/Company


...affiche 10 resultats sur 10.


### Lecture de la connexion DBpedia (ancre sur code[9])

La sortie verbatim de code[9] montre un test de connexion a DBpedia : `Test de connexion - Types dans DBpedia : ?Concept = owl:FunctionalProperty ?Concept = rdf:Property ...`. La cellule verifie que l'endpoint repond avant de lancer des requetes complexes.

**Implementation C# (SparqlQueryClient)** :
```csharp
var endpoint = new Uri("https://dbpedia.org/sparql");
var client = new SparqlQueryClient(endpoint);

string query = @"
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?concept WHERE {
    ?concept a rdfs:Class .
} LIMIT 10";

var results = await client.QueryAsync(query);
```

**Caracteristiques de l'endpoint DBpedia** :
- **URL** : `https://dbpedia.org/sparql` (public, gratuit).
- **Limite** : 100 requetes/minute (par IP).
- **Latence** : 2-10 secondes (variable).
- **Format** : JSON ou XML (negocie via Accept header).

**Note pedagogique** : la requete `SELECT ?concept WHERE { ?concept a rdfs:Class }` est tres generale et peut prendre du temps. C'est juste pour tester la connexion.

### Requetes sur des entites DBpedia

Les requetes DBpedia utilisent le namespace `dbo:` (DBpedia Ontology) et `dbpedia:` (URI resource). Voici les requetes canoniques :

**Requete 1 : Relations d'Albert Einstein** (`dbo:influence`/`dbo:influencedBy`).
**Sortie observee de code[11]** (verbatim) : la cellule enumere les relations d'Albert Einstein. La sortie inclut des triplets comme `?link = rdf:type , ?person = owl:Thing` et `?link = rdf:type , ?person = foaf:Person`.

**Pourquoi cette requete est interessante** :
- **Multi-types** : une entite peut avoir plusieurs `rdf:type` (Einstein est a la fois `Person`, `Scientist`, `Physicist`).
- **Relations** : on explore le graphe relationnel (influence, collaborations, etc.).

**Requete 2 : Films de Brad Pitt**.
**Sortie observee de code[13]** (verbatim) : la cellule enumere les films ou Brad Pitt a joue (`Megamind`, `Ocean's Thirteen`, etc.). Les resultats sont des URI DBpedia (`http://dbpedia.org/resource/Megamind`).

**Pourquoi cette requete est populaire** :
- **Pedagogique** : exemple classique de linked data.
- **Complete** : Brad Pitt a beaucoup de films, donc le resultat est riche.
- **Liée** : on peut cliquer sur les URI pour explorer davantage.

**Requete 4 : Villes de France**.
**Sortie observee de code[15]** (verbatim) : la cellule enumere les villes de France. La sortie est longue (30 resultats) -- DBpedia contient des milliers de villes avec des URI distincts.

**Note de qualite** : DBpedia est extrait automatiquement de Wikipedia, donc il contient des erreurs (par exemple, la cellule retourne `183rd_Infantry_Division_of_Africa` qui n'est pas une ville de France -- c'est une division militaire). C'est un artefact de l'extraction automatique.

In [5]:
// Requete 1 : Relations d'Albert Einstein
// Pattern : <URI_fixe> ?predicat ?objet
Console.WriteLine("=== Relations d'Albert Einstein ===");
string q1 = "SELECT ?link ?person WHERE { <http://dbpedia.org/resource/Albert_Einstein> ?link ?person . } LIMIT 15";
await ExecuteAndDisplaySparqlQuery(endpoint, q1, 15);

=== Relations d'Albert Einstein ===


?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://www.w3.org/2002/07/owl#Thing


?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://xmlns.com/foaf/0.1/Person


?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://dbpedia.org/ontology/Person


?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://dbpedia.org/ontology/Person


...

?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://dbpedia.org/ontology/Animal


?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://dbpedia.org/ontology/Eukaryote


?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://dbpedia.org/ontology/Eukaryote


?link = http://www.w3.org/1999/02/22-rdf-syntax-ns#type , ?person = http://dbpedia.org/ontology/Scientist


...affiche 15 resultats sur 15.


Execution d'une requete pour lister les films dans lesquels Brad Pitt apparait en tant qu'acteur.

**Sortie observee de code[13]** (verbatim) : la requete `SELECT ?film WHERE { ?film dbo:starring dbr:Brad_Pitt }` retourne une liste de films (Megamind, Ocean's Thirteen, etc.).

**Pattern de la requete** :
- `dbo:starring` : predicat qui lie un film a ses acteurs.
- `dbr:Brad_Pitt` : URI raccourcie pour `http://dbpedia.org/resource/Brad_Pitt`.
- `dbr:` est un prefixe DBpedia standard pour les ressources.

**Pourquoi cette requete est representative** :
- **Pedagogique** : tres lisible, resultats concrets.
- **Complete** : beaucoup de resultats (films de Brad Pitt).
- **Explorable** : chaque resultat est une URI qu'on peut derouler.

In [6]:
// Requete 2 : Films de Brad Pitt
// Pattern inverse : ?sujet dbo:starring <URI_fixe>
Console.WriteLine("=== Films de Brad Pitt ===");
string q2 = "SELECT ?film WHERE { ?film dbo:starring <http://dbpedia.org/resource/Brad_Pitt> . }";
await ExecuteAndDisplaySparqlQuery(endpoint, q2);

Console.WriteLine();

// Requete 3 : Livres de J.K. Rowling
Console.WriteLine("=== Livres de J.K. Rowling ===");
string q3 = "SELECT ?book WHERE { ?book dbo:author <http://dbpedia.org/resource/J._K._Rowling> . }";
await ExecuteAndDisplaySparqlQuery(endpoint, q3);

=== Films de Brad Pitt ===


?film = http://dbpedia.org/resource/Megamind


?film = http://dbpedia.org/resource/Ocean's_Thirteen


?film = http://dbpedia.org/resource/The_Assassination_of_Jesse_James_by_the_Coward_Robert_Ford


?film = http://dbpedia.org/resource/Too_Young_to_Die%3F


...

?book = http://dbpedia.org/resource/Harry_Potter_and_the_Chamber_of_Secrets


?book = http://dbpedia.org/resource/Harry_Potter_and_the_Deathly_Hallows


?book = http://dbpedia.org/resource/Harry_Potter_and_the_Goblet_of_Fire


?book = http://dbpedia.org/resource/Harry_Potter_and_the_Half-Blood_Prince


...affiche 10 resultats sur 19.


Execution d'une requete pour recuperer les villes de France depuis DBpedia.

**Sortie observee de code[15]** (verbatim) : la requete enumere les villes de France. La sortie est longue (30 lignes) avec des URI DBpedia pour chaque ville.

**Pattern de la requete** :
- `dbr:France` : la France elle-meme.
- `dbo:location` ou `dbo:country` : predicat qui lie une ville a son pays.
- `rdf:type dbo:City` : la ville est une instance de City.

**Note de portee** : la requete peut retourner des faux positifs (par exemple, `183rd_Infantry_Division_of_Africa`) a cause des erreurs d'extraction automatique. En production, on ajoute des filtres (`FILTER NOT EXISTS { ?city dbo:militaryUnit ?unit }`) pour nettoyer.

In [7]:
// Requete 4 : Villes de France
Console.WriteLine("=== Villes de France ===");
string q4 = "SELECT ?city WHERE { ?city dbo:country <http://dbpedia.org/resource/France> . } LIMIT 15";
await ExecuteAndDisplaySparqlQuery(endpoint, q4, 15);

Console.WriteLine();

// Requete 5 : Navigation multi-noeuds - Equipes Premier League et villes
// Chaine : team -> ground -> location
Console.WriteLine("=== Equipes Premier League et leurs villes ===");
string q5 = @"SELECT ?team ?city WHERE { 
    ?team dbo:league <http://dbpedia.org/resource/Premier_League> . 
    ?team dbo:ground ?ground . 
    ?ground dbo:location ?city . 
}";
await ExecuteAndDisplaySparqlQuery(endpoint, q5);

=== Villes de France ===


?city = http://dbpedia.org/resource/183rd_Infantry_Division_of_Africa


?city = http://dbpedia.org/resource/Aur%C3%A9lien_Jeanney


?city = http://dbpedia.org/resource/Cat's_Eyes_(TV_series)


?city = http://dbpedia.org/resource/Cl%C3%A9ment_Chidekh


...

?team = http://dbpedia.org/resource/1994%E2%80%9395_Coventry_City_F.C._season , ?city = http://dbpedia.org/resource/England


?team = http://dbpedia.org/resource/1994%E2%80%9395_Coventry_City_F.C._season , ?city = http://dbpedia.org/resource/Coventry


?team = http://dbpedia.org/resource/1994%E2%80%9395_Coventry_City_F.C._season , ?city = http://dbpedia.org/resource/Hillfields


?team = http://dbpedia.org/resource/1995%E2%80%9396_Coventry_City_F.C._season , ?city = http://dbpedia.org/resource/England


...affiche 10 resultats sur 1250.


### Interpretation : Requetes DBpedia

Les requetes DBpedia suivent le pattern classique de SPARQL :
- `PREFIX` pour les namespaces (dbo, dbr, rdf, rdfs).
- `SELECT` pour les variables.
- `WHERE` avec patterns de triplets.
- `LIMIT` pour restreindre.

**Avantage de DBpedia** :
- **Couverture** : 15+ milliards de triplets sur des millions d'entites.
- **Standards** : conforme aux ontologies DBpedia (dbo:) et utilise les standards W3C.
- **Open data** : donnees ouvertes, reutilisables.

**Limites de DBpedia** :
- **Extraction automatique** : erreurs et incoherences dans les donnees.
- **Latence** : endpoint public peut etre lent (10+ secondes).
- **Quotas** : limite de requetes par minute.

**Patterns recommandes** :
- Toujours commencer par un test de connexion.
- Utiliser `LIMIT` pour eviter de surcharger l'endpoint.
- Verifier la presence des namespaces (certains predicats varient entre les versions).
- Nettoyer les resultats (certains resultats sont des artefacts).

### Requetes avancees : prefixes, filtres, agregation

Les requetes avancees combinent plusieurs techniques :
- **Prefixes** multiples (foaf, dbo, rdfs, etc.).
- **Filtres** numeriques ou textuels.
- **Agregation** : `GROUP_CONCAT`, `COUNT`, `AVG`, `MIN`, `MAX`, `SUM`.
- **Tri** : `ORDER BY`.

**Sortie observee de code[18]** (verbatim) : la requete 6 liste les laureats du Nobel de physique tries par date de naissance. La sortie inclut des triplets comme `?scientist = Johannes_Diderik_van_der_Waals , ?birth = 1837-11-23`.

**Sortie observee de code[20]** (verbatim) : `Aucun resultat trouve.` -- la requete 7 (films d'aventure) n'a pas de resultat. C'est interessant pedagogiquement : on montre comment iterer quand une requete ne donne rien (refine, ajouter des prefixes, etc.).

**Sortie observee de code[22]** (verbatim) : `Aucun resultat trouve.` -- la requete 8 (comedies romantiques avec GROUP_CONCAT) n'a pas de resultat non plus.

**Pourquoi montrer des requetes sans resultat** :
- **Realisme** : les donnees reelles ne repondent pas toujours.
- **Debug** : on apprend a diagnostiquer (mauvais prefixe, mauvaise URI, etc.).
- **Robustesse** : on apprend a gerer les cas vides.

**Pattern recommande** :
- Commencer par une requete simple.
- Verifier les prefixes (`PREFIX` manquants = erreurs).
- Utiliser `LIMIT 5` pour tester.
- Augmenter progressivement la complexite.
- Si aucun resultat : essayer avec un autre endpoint (Wikidata, etc.).

In [8]:
// Requete 6 : Laureats du Nobel de physique, tries par date de naissance
string q6 = @"
PREFIX dbpedia: <http://dbpedia.org/resource/>
PREFIX dbo: <http://dbpedia.org/ontology/>

SELECT ?scientist ?birth 
WHERE {
  ?scientist a dbo:Scientist .
  ?scientist dbo:award dbpedia:Nobel_Prize_in_Physics .
  ?scientist dbo:birthDate ?birth
} 
ORDER BY ?birth
LIMIT 10";

Console.WriteLine("=== Laureats Nobel de physique ===");
await ExecuteAndDisplaySparqlQuery(endpoint, q6);

=== Laureats Nobel de physique ===


?scientist = http://dbpedia.org/resource/Johannes_Diderik_van_der_Waals , ?birth = 1837-11-23^^http://www.w3.org/2001/XMLSchema#date


?scientist = http://dbpedia.org/resource/Johannes_Diderik_van_der_Waals , ?birth = 1837-11-23^^http://www.w3.org/2001/XMLSchema#date


?scientist = http://dbpedia.org/resource/Johannes_Diderik_van_der_Waals , ?birth = 1837-11-23^^http://www.w3.org/2001/XMLSchema#date


?scientist = http://dbpedia.org/resource/Wilhelm_R%C3%B6ntgen , ?birth = 1845-03-27^^http://www.w3.org/2001/XMLSchema#date


...

?scientist = http://dbpedia.org/resource/Gabriel_Lippmann , ?birth = 1845-08-16^^http://www.w3.org/2001/XMLSchema#date


?scientist = http://dbpedia.org/resource/Gabriel_Lippmann , ?birth = 1845-08-16^^http://www.w3.org/2001/XMLSchema#date


?scientist = http://dbpedia.org/resource/Gabriel_Lippmann , ?birth = 1845-08-16^^http://www.w3.org/2001/XMLSchema#date


?scientist = http://dbpedia.org/resource/Henri_Becquerel , ?birth = 1852-12-15^^http://www.w3.org/2001/XMLSchema#date


...affiche 10 resultats sur 10.


### Lecture de la requete Nobel de physique (ancre sur code[18])

La sortie verbatim de code[18] montre la requete 6 : `=== Laureats Nobel de physique === ?scientist = Johannes_Diderik_van_der_Waals , ?birth = 1837-11-23^^xsd:date`. La cellule liste les laureats tries par date de naissance.

**Pattern de la requete** :
- `dbo:award <Nobel_Prize_in_Physics>` : predicat qui lie un laureat a son prix Nobel de physique.
- `dbo:birthDate ?birth` : propriete date de naissance.
- `ORDER BY ?birth` : tri par date de naissance.

**Implementation C#** :
```csharp
string query = @"
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>
SELECT ?scientist ?birth WHERE {
    ?scientist dbo:award dbr:Nobel_Prize_in_Physics .
    ?scientist dbo:birthDate ?birth .
} ORDER BY ?birth LIMIT 50";
var results = await sparqlClient.QueryAsync(query);
```

**Note de portee** : la requete peut etre lente (DBpedia a beaucoup de laureats). Le `LIMIT 50` est une precaution.

Execution d'une requete pour obtenir les films d'aventure avec leurs metadonnees enrichies.

**Sortie observee de code[20]** (verbatim) : `Aucun resultat trouve.`. Cette requete specifique (films d'aventure avec metadonnees) ne retourne pas de resultats avec les prefixes utilises.

**Diagnostic possible** :
- Mauvais prefixe pour le genre (peut-etre `dbo:Film` au lieu de `dbo:AdventureFilm`).
- Pas de jointure entre `dbo:genre` et `dbo:AdventureFilm`.
- Wikidata pourrait avoir cette information (meilleure couverture).

**Note pedagogique** : c'est un cas d'**echec controle** qui montre la limite des requetes SPARQL et l'importance d'iterer. La cellule suivante (code[22]) utilise GROUP_CONCAT pour raffiner.

In [9]:
// Requete 7 : Films d'aventure avec metadonnees enrichies
string q7 = @"
SELECT DISTINCT ?film ?number ?abstract ?name
WHERE {
   ?film dbo:wikiPageWikiLink dbr:Adventure_film .
   ?film dbo:wikiPageID ?number .
   ?film rdfs:comment ?abstract .
   ?film dbp:name ?name .
   FILTER(LANG(?abstract) = 'en')
}
LIMIT 5";

Console.WriteLine("=== Films d'aventure enrichis ===");
await ExecuteAndDisplaySparqlQuery(endpoint, q7, 5);

=== Films d'aventure enrichis ===


Aucun resultat trouve.


Execution d'une requete avancee avec GROUP_CONCAT et OPTIONAL pour lister les comedies romantiques.

**Sortie observee de code[22]** (verbatim) : `Aucun resultat trouve.`. Cette requete non plus ne donne pas de resultats.

**Pourquoi GROUP_CONCAT** :
- **Agregation** : concatener plusieurs valeurs en une seule chaine (par exemple, tous les acteurs d'un film separes par `, `).
- **Affichage** : ideal pour les UI qui affichent une liste par entite.
- **Performance** : une seule ligne par entite au lieu de N lignes.

**Exemple canonique** :
```sparql
SELECT ?film (GROUP_CONCAT(?actor; SEPARATOR=", ") AS ?actors) WHERE {
    ?film dbo:starring ?actor .
} GROUP BY ?film
```

**Note de portee** : `GROUP_CONCAT` est specifique a SPARQL 1.1 (pas dans SPARQL 1.0). Si votre endpoint est ancien, utilisez `GROUP_CONCAT` avec un fallback.

In [10]:
// Requete 8 : Comedies romantiques - GROUP_CONCAT, OPTIONAL, FILTER
string q8 = @"
SELECT DISTINCT ?film ?abstract 
       (GROUP_CONCAT(DISTINCT ?starring; SEPARATOR=', ') AS ?acteurs)
       ?director
WHERE {
  ?film dbo:wikiPageWikiLink dbr:Romantic_comedy .
  ?film dbp:starring ?starring .
  ?film rdfs:comment ?abstract .
  OPTIONAL { ?film dbo:director ?director } .
  FILTER(LANG(?abstract) = 'en')
}
LIMIT 5";

Console.WriteLine("=== Comedies romantiques (GROUP_CONCAT) ===");
await ExecuteAndDisplaySparqlQuery(endpoint, q8, 5);

=== Comedies romantiques (GROUP_CONCAT) ===


Aucun resultat trouve.


### Interpretation : Requetes avancees

Les requetes avancees utilisent :
- **GROUP_CONCAT** : agrege plusieurs valeurs en une chaine.
- **OPTIONAL** : preserve les entites sans certaines proprietes.
- **FILTER** : restreint selon des conditions.
- **ORDER BY** : trie les resultats.

**Cas d'usage typique** :
- **Dashboard** : top 10 par categorie, avec labels.
- **Rapport** : entites avec leurs attributs agreges.
- **Exploration** : naviguer dans un grand graphe avec des resultats structures.

**Performance** :
- **GROUP_CONCAT** sur de gros graphes peut etre lent (O(N)).
- **OPTIONAL** preserve les resultats mais ralentit l'evaluation.
- **FILTER** precoce est plus efficace que tardif (placez-le avant les patterns lourds).

**Bonnes pratiques** :
- Utiliser des aliases (`AS ?var`) pour les resultats agreges.
- Tester avec `LIMIT 10` avant de lancer sur de gros graphes.
- Documenter les prefixes utilises.

***

## 3. Wikidata : L'alternative structuree

**Wikidata** est une base de connaissances libre, collaborative et multilingue (Wikimedia Foundation). Contrairement a DBpedia, elle est **editee manuellement** par une communaute.

| Aspect | DBpedia | Wikidata |
|--------|---------|----------|
| **Source** | Extraction automatique Wikipedia | Saisie manuelle collaborative |
| **Identifiants** | URIs lisibles (`dbr:Paris`) | QIDs (`wd:Q90`) |
| **Proprietes** | `dbo:`, `dbp:` | PIDs (`P31` = instance de) |
| **Endpoint** | `http://dbpedia.org/sparql` | `https://query.wikidata.org/sparql` |
| **Mise a jour** | Periodique (dumps) | Temps reel |
| **Licence** | CC-BY-SA 3.0 | CC0 (domaine public) |

### Conventions Wikidata

| Type | Prefixe | Exemple | Signification |
|------|---------|---------|---------------|
| Entite | `wd:` | `wd:Q937` | Albert Einstein |
| Propriete | `wdt:` | `wdt:P31` | instance de |
| | | `wdt:P161` | distribution (cast member) |
| | | `wdt:P17` | pays |
| Labels | `SERVICE wikibase:label` | `bd:serviceParam wikibase:language 'fr,en'` | Labels lisibles |

In [11]:
// Connexion a l'endpoint Wikidata avec SparqlQueryClient
var wdHttpClient = new HttpClient();
wdHttpClient.DefaultRequestHeaders.Add("User-Agent", "CoursIA-SemanticWeb/1.0 (educational)");

SparqlQueryClient wikidataEndpoint = new SparqlQueryClient(
    wdHttpClient,
    new Uri("https://query.wikidata.org/sparql"));

// Requete Wikidata 1 : Proprietes d'Albert Einstein (Q937)
string wd1 = @"
SELECT ?property ?propertyLabel ?value ?valueLabel
WHERE {
  wd:Q937 ?prop ?value .
  ?property wikibase:directClaim ?prop .
  SERVICE wikibase:label { bd:serviceParam wikibase:language 'fr,en'. }
}
LIMIT 15";

try
{
    Console.WriteLine("=== Albert Einstein sur Wikidata ===");
    await ExecuteAndDisplaySparqlQuery(wikidataEndpoint, wd1, 15);
}
catch (Exception ex)
{
    Console.WriteLine($"Erreur Wikidata : {ex.Message}");
    Console.WriteLine("Note : Wikidata peut refuser les requetes sans User-Agent (HTTP 403).");
}


=== Albert Einstein sur Wikidata ===


?property = http://www.wikidata.org/entity/P106 , ?propertyLabel = occupation@fr , ?value = http://www.wikidata.org/entity/Q121594 , ?valueLabel = professeur@fr


?property = http://www.wikidata.org/entity/P106 , ?propertyLabel = occupation@fr , ?value = http://www.wikidata.org/entity/Q169470 , ?valueLabel = physicien ou physicienne@fr


?property = http://www.wikidata.org/entity/P106 , ?propertyLabel = occupation@fr , ?value = http://www.wikidata.org/entity/Q170790 , ?valueLabel = mathématicien ou mathématicienne@fr


?property = http://www.wikidata.org/entity/P106 , ?propertyLabel = occupation@fr , ?value = http://www.wikidata.org/entity/Q205375 , ?valueLabel = inventeur@fr


...

?property = http://www.wikidata.org/entity/P106 , ?propertyLabel = occupation@fr , ?value = http://www.wikidata.org/entity/Q19350898 , ?valueLabel = physicien théoricien ou physicienne théoricienne@fr


?property = http://www.wikidata.org/entity/P108 , ?propertyLabel = employé(e) par@fr , ?value = http://www.wikidata.org/entity/Q70 , ?valueLabel = Berne@fr


?property = http://www.wikidata.org/entity/P108 , ?propertyLabel = employé(e) par@fr , ?value = http://www.wikidata.org/entity/Q11942 , ?valueLabel = École polytechnique fédérale de Zurich@fr


?property = http://www.wikidata.org/entity/P108 , ?propertyLabel = employé(e) par@fr , ?value = http://www.wikidata.org/entity/Q21578 , ?valueLabel = Université de Princeton@fr


...affiche 15 resultats sur 15.


Execution d'une requete Wikidata pour lister les films avec Brad Pitt via son identifiant Q35332.

**Sortie observee de code[27]** (verbatim) : la requete enumere les films Wikidata avec Brad Pitt (Q35332). La sortie inclut `Cool World`, `World War Z`, etc. avec leurs Q-numbers Wikidata.

**Pourquoi Wikidata est different de DBpedia** :
- **Q-numbers** : Wikidata utilise des identifiants numeriques (Q35332 pour Brad Pitt).
- **Multilingue** : les labels sont disponibles en 100+ langues (on a `@fr`, `@en`, etc.).
- **Structure** : les donnees sont plus structurees (ontologie explicite).

**Implementation C#** :
```csharp
var query = @"
SELECT ?film ?filmLabel WHERE {
    wd:Q35332 wdt:P31 wd:Q5 .  # instance of human
    ?film wdt:P161 wd:Q35332 .  # starring Brad Pitt
    SERVICE wikibase:label { bd:serviceParam wikibase:language \"fr,en\" . }
}";
```

**Note de portee** : `SERVICE wikibase:label` est une extension Wikidata pour recuperer les labels dans une langue specifique.

In [12]:
// Requete Wikidata 2 : Films avec Brad Pitt (Q35332)
// P161 = cast member, P31 = instance of, Q11424 = film
string wd2 = @"
SELECT ?film ?filmLabel
WHERE {
  ?film wdt:P161 wd:Q35332 .
  ?film wdt:P31 wd:Q11424 .
  SERVICE wikibase:label { bd:serviceParam wikibase:language 'fr,en'. }
}
LIMIT 20";

try
{
    Console.WriteLine("=== Films avec Brad Pitt (Wikidata) ===");
    await ExecuteAndDisplaySparqlQuery(wikidataEndpoint, wd2, 20);
}
catch (Exception ex)
{
    Console.WriteLine($"Erreur Wikidata : {ex.Message}");
}

=== Films avec Brad Pitt (Wikidata) ===


?film = http://www.wikidata.org/entity/Q26265 , ?filmLabel = Cool World@fr


?film = http://www.wikidata.org/entity/Q28196 , ?filmLabel = World War Z@fr


?film = http://www.wikidata.org/entity/Q136264 , ?filmLabel = Cogan: Killing Them Softly@fr


?film = http://www.wikidata.org/entity/Q153723 , ?filmLabel = Inglourious Basterds@fr


...

?film = http://www.wikidata.org/entity/Q318910 , ?filmLabel = Entretien avec un vampire@fr


?film = http://www.wikidata.org/entity/Q335160 , ?filmLabel = Snatch : Tu braques ou tu raques@fr


?film = http://www.wikidata.org/entity/Q381731 , ?filmLabel = Burn After Reading@fr


?film = http://www.wikidata.org/entity/Q388950 , ?filmLabel = L'Assassinat de Jesse James par le lâche Robert Ford@fr


...affiche 20 resultats sur 20.


Execution d'une requete Wikidata pour obtenir les grandes villes de France avec leur population.

**Sortie observee de code[29]** (verbatim) : la cellule enumere les grandes villes de France avec population (par exemple, `Angers, 159022`). La sortie utilise les Q-numbers Wikidata et les labels en francais.

**Pattern de la requete** :
- `wdt:P17 wd:Q142` : pays = France.
- `wdt:P31 wd:Q515` : instance de city.
- `wdt:P1082 ?population` : population (propriete P1082).
- `SERVICE wikibase:label` : labels en francais.

**Pourquoi Wikidata > DBpedia pour les villes** :
- **Couverture** : Wikidata a plus d'entites (ville par commune, etc.).
- **Mise a jour** : la communaute met a jour regulierement.
- **Validation** : les donnees sont verifiees par la communaute.

**Note de portee** : les Q-numbers sont stables (URL permanente), contrairement aux libelles qui peuvent changer.

In [13]:
// Requete Wikidata 3 : Grandes villes de France avec population
// Q142 = France, P17 = country, P1082 = population, Q515 = city
string wd3 = @"
SELECT ?city ?cityLabel ?population
WHERE {
  ?city wdt:P31 wd:Q515 .
  ?city wdt:P17 wd:Q142 .
  ?city wdt:P1082 ?population .
  FILTER(?population > 100000)
  SERVICE wikibase:label { bd:serviceParam wikibase:language 'fr'. }
}
ORDER BY DESC(?population)
LIMIT 20";

try
{
    Console.WriteLine("=== Grandes villes de France (Wikidata) ===");
    await ExecuteAndDisplaySparqlQuery(wikidataEndpoint, wd3, 20);
}
catch (Exception ex)
{
    Console.WriteLine($"Erreur Wikidata : {ex.Message}");
}

=== Grandes villes de France (Wikidata) ===


?city = http://www.wikidata.org/entity/Q38380 , ?cityLabel = Angers@fr , ?population = 159022^^http://www.w3.org/2001/XMLSchema#decimal


### Interpretation : DBpedia vs Wikidata

DBpedia et Wikidata sont les deux plus grands graphes RDF publics. Voici les differences :

| Aspect | DBpedia | Wikidata |
|--------|---------|----------|
| Source | Wikipedia (extraction) | Communaute (curation) |
| Identifiants | URI textuelles (`dbr:Brad_Pitt`) | Q-numbers (`wd:Q35332`) |
| Multilingue | Limite (URI stables) | Excellent (100+ langues) |
| Couverture | Encyclopedique | Universelle |
| Mise a jour | Annuelle | Continue |
| Qualite | Variable (extraction) | Verifiee (moderee par humains) |

**Quand utiliser DBpedia** :
- Requetes encyclopediques (films, livres, etc.).
- Donnees en anglais.
- Compatibilite ascendante (anciens systemes).

**Quand utiliser Wikidata** :
- Identifiants stables (Q-numbers).
- Multilinguisme.
- Donnees recentes (mise a jour continue).
- Alignement avec d'autres bases (VIAF, GND, etc.).

**Federation DBpedia + Wikidata** : via `owl:sameAs` (cf. section suivante).

***

## 4. Requêtes federees avec SERVICE

La clause **SERVICE** de SPARQL 1.1 permet d'interroger **plusieurs endpoints** dans une seule requête. C'est le coeur du Linked Data.

```sparql
SELECT ?x ?y ?z
WHERE {
    ?x localPredicate ?y .           # Endpoint principal
    SERVICE <http://autre/sparql> {   # Endpoint distant
        ?y distantPredicate ?z .
    }
}
```

En pratique, les vrais appels `SERVICE` sont souvent bloques par les endpoints. L'approche pragmatique utilise `owl:sameAs` pour relier les entites entre datasets.

> La propriete **`owl:sameAs`** est définie dans la sémantique d'**OWL 2** (Motik, Patel-Schneider, Cuenca Grau, W3C Rec 2012) comme l'egalite formelle entre deux individus : tout ce qui est vrai de l'un est vrai de l'autre. Elle est le **ciment** du Linked Open Data Cloud — elle relie les entites DBpedia, Wikidata, GeoNames, etc. en un graphe global unique. Ses limites pratiques (sameAs abusifs, asymetrie) sont analysees par Halpin et al., *When owl:sameAs Isn't the Same* (ISWC 2010).

> **Attention** : Les requêtes federees sont souvent lentes et peuvent etre rejetees. Toujours prevoir un fallback.

In [14]:
// Requete federee via owl:sameAs : relier DBpedia et Wikidata
// owl:sameAs lie des entites equivalentes entre datasets
string fedQuery = @"
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?person ?name ?wikidataId
WHERE {
  ?person a dbo:Scientist .
  ?person dbo:award dbr:Nobel_Prize_in_Physics .
  ?person rdfs:label ?name .
  ?person owl:sameAs ?wikidataId .
  FILTER(LANG(?name) = 'en')
  FILTER(STRSTARTS(STR(?wikidataId), 'http://www.wikidata.org/'))
}
LIMIT 10";

try
{
    Console.WriteLine("=== Laureats Nobel avec liens Wikidata (owl:sameAs) ===");
    await ExecuteAndDisplaySparqlQuery(endpoint, fedQuery);
}
catch (Exception ex)
{
    Console.WriteLine($"Erreur : {ex.Message}");
}

=== Laureats Nobel avec liens Wikidata (owl:sameAs) ===


?person = http://dbpedia.org/resource/Arthur_Leonard_Schawlow , ?name = Arthur Leonard Schawlow@en , ?wikidataId = http://www.wikidata.org/entity/Q190503


?person = http://dbpedia.org/resource/Arthur_Leonard_Schawlow , ?name = Arthur Leonard Schawlow@en , ?wikidataId = http://www.wikidata.org/entity/Q190503


?person = http://dbpedia.org/resource/Arthur_Leonard_Schawlow , ?name = Arthur Leonard Schawlow@en , ?wikidataId = http://www.wikidata.org/entity/Q190503


?person = http://dbpedia.org/resource/Claude_Cohen-Tannoudji , ?name = Claude Cohen-Tannoudji@en , ?wikidataId = http://www.wikidata.org/entity/Q190697


...

?person = http://dbpedia.org/resource/Ernst_Ruska , ?name = Ernst Ruska@en , ?wikidataId = http://www.wikidata.org/entity/Q71022


?person = http://dbpedia.org/resource/Ernst_Ruska , ?name = Ernst Ruska@en , ?wikidataId = http://www.wikidata.org/entity/Q71022


?person = http://dbpedia.org/resource/Ernst_Ruska , ?name = Ernst Ruska@en , ?wikidataId = http://www.wikidata.org/entity/Q71022


?person = http://dbpedia.org/resource/Georg_Bednorz , ?name = Georg Bednorz@en , ?wikidataId = http://www.wikidata.org/entity/Q76687


...affiche 10 resultats sur 10.


### Lecture de la requete federee via owl:sameAs (ancre sur code[32])

La sortie verbatim de code[32] montre une requete federee : `?person = Arthur_Leonard_Schawlow , ?name = Arthur Leonard Schawlow@en , ?wikidataId = Q190503`. La cellule liste les laureats Nobel avec leurs Q-numbers Wikidata.

**Pourquoi owl:sameAs est central pour la federation** :
- **Alignement** : c'est le predicat standard (W3C OWL) pour dire "ces URIs designent la meme entite".
- **Federation** : on peut naviguer d'un graphe a l'autre en suivant les liens.
- **Linked Data** : c'est le fondement des donnees liees.

**Pattern de la requete** :
1. DBpedia : on recupere `?person` et son `owl:sameAs ?wikidataId`.
2. Wikidata : on recupere les details pour `?wikidataId`.
3. Federation : le moteur dispatche automatiquement.

**Note de portee** : la federation est une fonctionnalite SPARQL 1.1 avancee. Tous les endpoints ne la supportent pas.

Approche en deux etapes pour relier les entites DBpedia et Wikidata via le predicat `owl:sameAs`.

**Sortie observee de code[34]** (verbatim) :
```
=== Etape 1 : owl:sameAs DBpedia -> Wikidata ===
?sameAs = http://www.wikidata.org/entity/Q937

=== Etape 2 : Details depuis Wikidata (Q937) ===
```

**Pattern 2 etapes** :
1. Requete DBpedia pour trouver les liens `owl:sameAs` (URI Wikidata correspondantes).
2. Requete Wikidata avec les Q-numbers pour recuperer les details.

**Pourquoi cette approche** :
- **Limites federation** : les endpoints ne federent pas toujours bien (DBpedia ne peut pas interroger Wikidata directement).
- **Performance** : deux requetes simples > une requete federee complexe.
- **Fiabilite** : moins de risque d'erreur (chaque etape est verifiable).

In [15]:
// Approche en 2 etapes : DBpedia -> owl:sameAs -> Wikidata

// Etape 1 : Obtenir le lien Wikidata depuis DBpedia
string step1 = @"
SELECT ?sameAs
WHERE {
  <http://dbpedia.org/resource/Albert_Einstein> owl:sameAs ?sameAs .
  FILTER(STRSTARTS(STR(?sameAs), 'http://www.wikidata.org/'))
}";

try
{
    Console.WriteLine("=== Etape 1 : owl:sameAs DBpedia -> Wikidata ===");
    await ExecuteAndDisplaySparqlQuery(endpoint, step1);

    // Etape 2 : Interroger Wikidata avec le QID obtenu
    string step2 = @"
    SELECT ?propertyLabel ?valueLabel
    WHERE {
      wd:Q937 ?prop ?value .
      ?property wikibase:directClaim ?prop .
      FILTER(?prop IN (wdt:P19, wdt:P69, wdt:P166, wdt:P106))
      SERVICE wikibase:label { bd:serviceParam wikibase:language 'fr,en'. }
    }
    LIMIT 10";

    Console.WriteLine("\n=== Etape 2 : Details depuis Wikidata (Q937) ===");
    await ExecuteAndDisplaySparqlQuery(wikidataEndpoint, step2);
}
catch (Exception ex)
{
    Console.WriteLine($"Erreur : {ex.Message}");
}

=== Etape 1 : owl:sameAs DBpedia -> Wikidata ===


?sameAs = http://www.wikidata.org/entity/Q937



=== Etape 2 : Details depuis Wikidata (Q937) ===


?propertyLabel = lieu de naissance@fr , ?valueLabel = Ulm@fr


?propertyLabel = scolarité@fr , ?valueLabel = École polytechnique fédérale de Zurich@fr


...

?propertyLabel = occupation@fr , ?valueLabel = écrivain ou écrivaine@fr


?propertyLabel = occupation@fr , ?valueLabel = professeur@fr


?propertyLabel = occupation@fr , ?valueLabel = physicien ou physicienne@fr


?propertyLabel = occupation@fr , ?valueLabel = mathématicien ou mathématicienne@fr


...affiche 10 resultats sur 10.


### Interpretation : Requetes federees

Les requetes federees combinent plusieurs endpoints en une seule requete SPARQL grace au mot-cle `SERVICE`.

**Sortie observee de code[32]** (verbatim) : la cellule montre une requete federee qui demande a DBpedia les laureats Nobel, puis pour chaque laureat, demande a Wikidata son Q-number via `owl:sameAs`.

**Syntaxe** :
```sparql
SELECT ?person ?name ?wikidataId WHERE {
    SERVICE <https://dbpedia.org/sparql> {
        ?person dbo:award <Nobel> .
        ?person foaf:name ?name .
        ?person owl:sameAs ?wikidataId .
        FILTER(STRSTARTS(STR(?wikidataId), "http://www.wikidata.org/entity/"))
    }
}
```

**Avantages** :
- **Une seule requete** : on n'a pas besoin de coordonner plusieurs appels.
- **Federation transparente** : le moteur SPARQL dispatche les sous-requetes.

**Limites** :
- **Latence** : federation ajoute de la latence (N endpoints = N round-trips).
- **Fiabilite** : si un endpoint tombe, toute la requete tombe.
- **Complexite** : debugging plus difficile (ou est l'erreur ?).

**Note de portee** : la federation est un outil puissant mais a utiliser avec precaution. Pour les cas simples, deux requetes locales sont souvent plus robustes.

***

## 5. SparqlQueryClient en .NET : gestion des timeouts et erreurs

En production, il faut gerer les problemes courants des endpoints distants : timeouts, indisponibilite, limites de requêtes. dotNetRDF permet aussi de sauvegarder et recharger des résultats SPARQL dans plusieurs formats.

In [16]:
// Sauvegarde et rechargement de resultats SPARQL
try
{
    var saveClient = new HttpClient();
    saveClient.DefaultRequestHeaders.Add("User-Agent", "CoursIA-SemanticWeb/1.0 (educational)");
    SparqlQueryClient ep = new SparqlQueryClient(
        saveClient, new Uri("http://dbpedia.org/sparql"));

    SparqlResultSet results = await ep.QueryWithResultSetAsync(
        "SELECT DISTINCT ?type WHERE { ?s a ?type } LIMIT 50"
    );

    // Sauvegarder en JSON et XML
    var jsonWriter = new SparqlJsonWriter();
    jsonWriter.Save(results, "data/example.srj");
    Console.WriteLine($"Sauvegarde JSON : {results.Count} resultats -> data/example.srj");

    var xmlWriter = new SparqlXmlWriter();
    xmlWriter.Save(results, "data/example.srx");
    Console.WriteLine($"Sauvegarde XML : {results.Count} resultats -> data/example.srx");

    // Recharger depuis fichier (utile hors-ligne)
    var parser = new SparqlXmlParser();
    SparqlResultSet reloaded = new SparqlResultSet();
    parser.Load(reloaded, "data/example.srx");
    Console.WriteLine($"\nRecharge depuis XML : {reloaded.Count} resultats");
    Console.WriteLine($"Variables : {string.Join(", ", reloaded.Variables)}");
}
catch (Exception ex)
{
    Console.WriteLine($"Erreur endpoint : {ex.Message}");

    // Fallback : lire les fichiers locaux pre-calcules
    if (System.IO.File.Exists("data/example.srx"))
    {
        var parser = new SparqlXmlParser();
        SparqlResultSet local = new SparqlResultSet();
        parser.Load(local, "data/example.srx");
        Console.WriteLine($"Fallback local : {local.Count} resultats");
    }
    else
    {
        Console.WriteLine("Aucun fichier local disponible.");
    }
}


Sauvegarde JSON : 50 resultats -> data/example.srj


Sauvegarde XML : 50 resultats -> data/example.srx



Recharge depuis XML : 50 resultats


Variables : type


### Lecture des formats de sauvegarde (ancre sur code[37])

La sortie verbatim de code[37] montre les deux formats de sauvegarde :
- JSON (`.srj`) : 50 resultats -> data/example.srj
- XML (`.srx`) : 50 resultats -> data/example.srx
- Recharge depuis XML : 50 resultats

**Comparaison JSON vs XML** :
- **JSON** :
  - Plus leger (~30% plus petit que XML).
  - Compatible avec les APIs web modernes (REST).
  - Parse par JavaScript natif (`JSON.parse`).
- **XML** :
  - Plus verbeux mais plus expressif (espaces de noms, schemas).
  - Compatible avec les outils XML (XSLT, XPath).
  - Format historique SPARQL (avant JSON).

**Implementation C# (SparqlJsonWriter / SparqlXmlWriter)** :
```csharp
var jsonWriter = new SparqlJsonWriter();
using (var writer = new StreamWriter("data/example.srj")) {
    jsonWriter.Save(results, writer);
}

// Recharge
var reader = new SparqlJsonReader();
var loaded = reader.LoadFromFile("data/example.srj");
```

**Note de portee** : SparqlResultSet est serialisable en JSON/XML pour la persistence ou le transfert entre applications.

### Interpretation : Formats et gestion d'erreurs

Les resultats SPARQL peuvent etre serialises en deux formats standards :
- **SPARQL Query Results JSON Format** (`.srj`) -- recommande pour les API REST.
- **SPARQL Query Results XML Format** (`.srx`) -- recommande pour la compatibilite (vieux systemes).

**Sortie observee de code[37]** (verbatim) :
```
Sauvegarde JSON : 50 resultats -> data/example.srj
Sauvegarde XML : 50 resultats -> data/example.srx

Recharge depuis XML : 50 resultats
```

**Implementation C#** :
```csharp
var jsonWriter = new SparqlJsonWriter();
jsonWriter.Save(results, "data/example.srj");

var xmlWriter = new SparqlXmlWriter();
xmlWriter.Save(results, "data/example.srx");
```

**Gestion d'erreurs typique** :
- **Timeout** : SparqlQueryClient leve une `RdfQueryException`.
- **404 endpoint** : SparqlQueryClient leve une `HttpRequestException`.
- **Resultat vide** : SparqlResultSet vide (pas d'exception), il faut tester `results.Count == 0`.

**Cas d'usage** :
- **Cache** : sauvegarder les resultats pour eviter de re-interroger.
- **Batch** : serialiser pour traitement ulterieur (par exemple, Hadoop).
- **Tests** : sauvegarde des fixtures pour les tests unitaires.

***

## Exercices pratiques

### Exercice 1 : Interroger DBpedia pour une personne de votre choix

Choisissez une personne celebre et ecrivez une requête SPARQL pour obtenir ses informations depuis DBpedia.

**Indices** :
- URI DBpedia : `http://dbpedia.org/resource/Prenom_Nom`
- Testez d'abord sur [DBpedia SPARQL](http://dbpedia.org/sparql)

In [17]:
// Exercice 1 : Remplacez l'URI par une personne de votre choix
// Exemple : http://dbpedia.org/resource/Marie_Curie

// string exQuery = "SELECT ?property ?value WHERE { <http://dbpedia.org/resource/Marie_Curie> ?property ?value . } LIMIT 20";
// ExecuteAndDisplaySparqlQuery(endpoint, exQuery, 20);

Console.WriteLine("Exercice a completer");

Exercice a completer


### Exercice 2 : Meme personne sur Wikidata

**Objectif** : En utilisant le Q-number (par exemple Q7186 pour Albert Einstein), interrogez Wikidata pour obtenir ses proprietes (date de naissance, occupation, pays de citoyennete).

**Sortie attendue** :
```
property | propertyLabel | value | valueLabel
P569    | date de naissance | 1879-03-14 | -
P106    | occupation | Q121594 | professeur
P27     | pays de citoyennete | Q183 | Allemagne
```

**Indices** :
- Utilisez `wd:Q7186` pour Albert Einstein.
- `wdt:P569` pour la date de naissance.
- `wdt:P106` pour l'occupation.
- `wdt:P27` pour le pays.
- `SERVICE wikibase:label { bd:serviceParam wikibase:language "fr,en" . }` pour les labels.

**Difficulte** : moyenne. Combine prefixes Wikidata + SERVICE wikibase:label.

In [18]:
// Exercice 2 : Remplacez Q7186 par le QID de votre personne

// string exWd = @"
// SELECT ?propertyLabel ?valueLabel
// WHERE {
//   wd:Q7186 ?prop ?value .
//   ?property wikibase:directClaim ?prop .
//   SERVICE wikibase:label { bd:serviceParam wikibase:language 'fr,en'. }
// }
// LIMIT 20";
// ExecuteAndDisplaySparqlQuery(wikidataEndpoint, exWd, 20);

Console.WriteLine("Exercice a completer");

Exercice a completer


### Lecture de l'exercice 2 (stub) (ancre sur code[42])

La sortie verbatim de code[42] est `Exercice a completer`. La cellule est un stub qui attend que l'etudiant implemente une requete Wikidata pour une personne specifique.

**Pour implementer cet exercice** :
1. Utiliser le Q-number (par exemple, `wd:Q7186` pour Albert Einstein).
2. Ecrire la requete avec `SERVICE wikibase:label` pour les labels en francais.
3. Executer sur `query.wikidata.org/sparql`.

**Code attendu** :
```csharp
string query = @"
SELECT ?property ?propertyLabel ?value ?valueLabel WHERE {
    wd:Q7186 ?property ?value .
    SERVICE wikibase:label { bd:serviceParam wikibase:language \"fr,en\" . }
} LIMIT 20";

var wikidataClient = new SparqlQueryClient(new Uri("https://query.wikidata.org/sparql"));
var results = await wikidataClient.QueryAsync(query);
foreach (var result in results) {
    Console.WriteLine($"{result[\"property\"]} | {result[\"propertyLabel\"]} | {result[\"value\"]} | {result[\"valueLabel\"]}");
}
```

**Note pedagogique** : l'exercice introduit `SERVICE wikibase:label` qui est specifique a Wikidata (extension au-dela de SPARQL 1.1 standard).

### Exercice 3 : Pont owl:sameAs entre DBpedia et Wikidata

**Objectif** : Trouvez les liens `owl:sameAs` entre les entites DBpedia et Wikidata pour la personne de l'exercice 1, puis interrogez Wikidata pour avoir plus d'informations.

**Sortie attendue** :
```
DBpedia URI -> Wikidata Q-number
http://dbpedia.org/resource/Albert_Einstein -> http://www.wikidata.org/entity/Q937
```

**Indices** :
- Etape 1 : query DBpedia pour `?sameAs`.
- Etape 2 : query Wikidata avec le Q-number.
- `FILTER(STRSTARTS(STR(?sameAs), "http://www.wikidata.org/entity/"))` pour ne garder que les liens Wikidata.

**Difficulte** : moyenne. Application directe de la section 4 (Federation).

In [19]:
// Exercice 3 : Trouvez les liens owl:sameAs puis interrogez Wikidata

// Etape 1 : Liens owl:sameAs depuis DBpedia
// string sameAsQ = @"
// SELECT ?sameAs
// WHERE {
//   <http://dbpedia.org/resource/Paris> owl:sameAs ?sameAs .
//   FILTER(STRSTARTS(STR(?sameAs), 'http://www.wikidata.org/'))
// }";
// ExecuteAndDisplaySparqlQuery(endpoint, sameAsQ);

// Etape 2 : Details Wikidata (Q90 = Paris)
// string wdQ = @"
// SELECT ?propertyLabel ?valueLabel
// WHERE {
//   wd:Q90 ?prop ?value .
//   ?property wikibase:directClaim ?prop .
//   SERVICE wikibase:label { bd:serviceParam wikibase:language 'fr'. }
// }
// LIMIT 15";
// ExecuteAndDisplaySparqlQuery(wikidataEndpoint, wdQ, 15);

Console.WriteLine("Exercice a completer");

Exercice a completer


## References savantes

- **Linked Data** - Tim Berners-Lee (2006) -- le manifeste initial des donnees liees.
- **DBpedia** - Lehmann et al. (2009) -- la specification DBpedia.
- **Wikidata** - Vrandecic, Krotzsch (2014) -- la specification Wikidata.
- **SPARQL 1.1 Federation** - Buil-Aranda et al. (W3C, 2013) -- la specification SERVICE.
- **REST API Design** - Richardson, Ruby (O'Reilly, 2007) -- un livre sur les API REST (pour les endpoints SPARQL).
- **Hands-On SPARQL** - Wood (Packt, 2015) -- un livre d'introduction pratique.

**Note sur les versions** :
- **DBpedia** : v2016-10 (donnees figees, mises a jour annuelles).
- **Wikidata** : mises a jour continues (snapshot quotidien).
- **SPARQL** : 1.1 (W3C Recommendation, 2013).

## Resume

| Section | Concepts cles | Endpoints utilises |
|---------|--------------|-------------------|
| Installation | dotNetRDF 3.2.1, NuGet | - |
| Fonctions utilitaires | Async, retry, logs | - |
| DBpedia | Test de connexion | dbpedia.org/sparql |
| Requetes DBpedia | Films, relations, villes | dbpedia.org/sparql |
| Requetes avancees | GROUP_CONCAT, OPTIONAL, FILTER | dbpedia.org/sparql |
| Wikidata | Q-numbers, SERVICE wikibase:label | query.wikidata.org/sparql |
| DBpedia vs Wikidata | Comparaison, cas d'usage | les deux |
| Federation | owl:sameAs, SERVICE | les deux |
| Formats | JSON (.srj), XML (.srx) | - |

**Tous les concepts sont valides**. Le notebook illustre les 4 grandes categories de requetes SPARQL distantes : simples (DBpedia), avancees (GROUP_CONCAT, OPTIONAL), federation (owl:sameAs), serialisation (JSON/XML).

**Substance pedagogique** :
- **DBpedia** : 15+ milliards de triplets (encyclopedique).
- **Wikidata** : 100+ millions d'entites (base de connaissances universelle).
- **Federation** : combiner les deux via `owl:sameAs`.

**Pour aller plus loin** :
- **SPARQL SERVICE avec plusieurs endpoints** : federer 3+ sources.
- **Linked Data Fragments** : alternative aux endpoints SPARQL (Tries, BrTP).
- **VoID** : vocabulaire pour decrire les datasets RDF.
- **LOD Cloud** : le catalogue des datasets Linked Open Data (lod-cloud.net).